# DATN — Train phát triển sau đợt thu tối đa3 ngày

Dùng bộ mới **48 run,12 mỗi nhóm**, ba ngày UTC thật, đủ Account/C2 và hai nhãn
mỗi ngày. Tất cả là dữ liệu TRAIN/phát triển. Kiểm định chéo giữ nguyên nhóm
máy/ngày, không chia ngẫu nhiên từng dòng để tạo độc lập giả.

Không có validation/test độc lập trong phạm vi này. CV theo ngày đánh giá quy
trình chọn model +calibration ở quy mô nhỏ, không phải đánh giá tương lai theo
thứ tự thời gian. Threshold chọn trên OOF là kết quả lựa chọn phát triển.
CI bên ngoài để null, uncertainty chưa đủ; deployment/automation luôn false.

CPU là đủ. Chỉ chạy sau khi Thịnh audit raw và commit **đúng manifest READY của
bộ48 run** lên GitHub. Candidate COLLECTION_READY chỉ cho thu dữ liệu, không cho
train một dataset chưa kiểm. Giữ nguyên dữ liệu/tags/bộ216 cũ. Artifact mới là
offline development_model.joblib, không phải package production để freeze/API.


## 0. Cấu hình

Upload đúng3file từ lệnh stage vào
MyDrive/DATN/development-3day-data/processed-development-v1:
train.jsonl, dataset_manifest.json, colab-upload.sha256.
Không upload raw, validation, locked test, .env hoặc khóa. Sao chép notebook vào
Drive để lưu output; điền SHA review READY do Thịnh bàn giao rồi dùng runtime mới.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/lethinhlord/Repo.git"
BRANCH = "main"  # Transport only; the owner-approved detached SHA is authoritative.
EXPECTED_TRAINING_COMMIT = "0fcacb145146eb9657739e76b56a870f75818ac1"
PROJECT_DIR = Path("/content/datn-ai-soar")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/DATN/development-3day-data/processed-development-v1")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/DATN/development-3day-results")
EXPECTED_DATA_FILES = ("train.jsonl", "dataset_manifest.json")
CHECKSUM_FILE = "colab-upload.sha256"
EXPECTED_FILES = (*EXPECTED_DATA_FILES, CHECKSUM_FILE)
RANDOM_SEED = 42
print("Source transport:", REPO_URL, BRANCH)
print("Development data:", DRIVE_DATA_DIR)

## 1. Kiểm source và manifest trước cài thư viện

SHA phải là commit review READY cho bộ48 run, không tự lấy SHA builder từ
manifest làm quyền chạy code. Nếu chưa có review đó, dừng và bàn giao raw cho
Thịnh audit. Không bỏ guard hoặc dùng manifest của bộ40/216 để thay thế.


In [ ]:
import hashlib
import importlib
import json
import os
import platform
import re
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
if not re.fullmatch(r"[0-9a-f]{40}", EXPECTED_TRAINING_COMMIT):
    raise RuntimeError("Set the owner-approved40-character development READY review SHA")
if PROJECT_DIR.exists():
    raise RuntimeError("Use a fresh runtime; an existing source checkout is refused")
for name in ("validation.jsonl", "locked_test.jsonl", ".env"):
    if (DRIVE_DATA_DIR / name).exists():
        raise RuntimeError("Forbidden development staging member: " + name)
missing = [name for name in EXPECTED_FILES if not (DRIVE_DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing development staging files: {missing}")
expected_hashes = {}
for line in (DRIVE_DATA_DIR / CHECKSUM_FILE).read_text().splitlines():
    if not line.strip():
        continue
    digest, name = line.split()
    name = name.lstrip("*")
    if name in expected_hashes or re.fullmatch(r"[0-9a-f]{64}", digest) is None:
        raise RuntimeError("Invalid/duplicate development checksum")
    expected_hashes[name] = digest
if set(expected_hashes) != set(EXPECTED_DATA_FILES):
    raise RuntimeError("Checksum must cover exactly TRAIN and development manifest")
for name in EXPECTED_DATA_FILES:
    if hashlib.sha256((DRIVE_DATA_DIR / name).read_bytes()).hexdigest() != expected_hashes[name]:
        raise RuntimeError("Staging checksum mismatch: " + name)
staged_manifest = json.loads((DRIVE_DATA_DIR / "dataset_manifest.json").read_bytes())
if staged_manifest.get("manifest_version") != "dataset-manifest-development-3day-v1":
    raise RuntimeError("This notebook accepts only the reviewed48-run development dataset")
# git clone --branch main is transport; the explicit SHA is never taken from untrusted data.
subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "checkout", "--detach", EXPECTED_TRAINING_COMMIT], check=True
)
os.chdir(PROJECT_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if commit != EXPECTED_TRAINING_COMMIT:
    raise RuntimeError("Development source differs from owner-approved SHA")
sys.path.insert(0, str(PROJECT_DIR))
source_verifier = importlib.import_module("ml.development_sources")
source_verifier.verify_manifest(staged_manifest, root=PROJECT_DIR, expected_commit=commit)
print("Verified development source/manifest:", commit, source_verifier.digest(staged_manifest))

## 2. Môi trường Python3.12 cô lập

Kernel gắn Drive/hiển thị; cùng một Python3.12.15 cô lập sẽ kiểm dữ liệu, chạy
quality gate và train. Giữ lock/hash/constraint giống runtime đã kiểm. Log ghi
thời điểm bắt đầu/kết thúc và output thật của từng tiến trình; pytest hiển thị
tên test đang chạy. Tiến trình im lặng không có heartbeat định kỳ.


In [ ]:
import shutil

from ml.command_logging import run_logged

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("This bootstrap requires the standard x86_64 Linux Colab CPU runtime")
EVIDENCE_DIR = PROJECT_DIR / "operator-logs" / "development-colab"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
SESSION_LOG = EVIDENCE_DIR / "development-colab-training.log"
TRAIN_ENV = PROJECT_DIR.parent / "development-training-venv"
UV_TOOLS = PROJECT_DIR.parent / "development-uv-bootstrap"
if TRAIN_ENV.exists() or UV_TOOLS.exists():
    raise RuntimeError("Existing training/tools environment: start a fresh Colab runtime")
TRAIN_PYTHON = TRAIN_ENV / "bin" / "python"
UV = UV_TOOLS / "bin" / "uv"

# Official PyPI uv 0.13.9 Linux x86_64 wheel. Keep the bootstrap out of the Colab kernel.
uv_spec = EVIDENCE_DIR / "uv-bootstrap.txt"
uv_spec.write_text(
    "uv==0.13.9 --hash=sha256:d7f459fe4fd247d830351c75686d3996ace3c6f3f728eb0e85a30b7ada0cef19\n",
    encoding="utf-8",
)
run_logged(
    "Install verified uv bootstrap",
    [
        sys.executable,
        "-m",
        "pip",
        "--isolated",
        "install",
        "--index-url",
        "https://pypi.org/simple",
        "--only-binary=:all:",
        "--no-deps",
        "--require-hashes",
        "--target",
        UV_TOOLS,
        "-r",
        uv_spec,
    ],
    SESSION_LOG,
)
run_logged(
    "Create Python 3.12 environment",
    [
        UV,
        "venv",
        "--managed-python",
        "--python",
        "3.12.15",
        "--seed",
        TRAIN_ENV,
    ],
    SESSION_LOG,
)
train_env = {
    **os.environ,
    "VIRTUAL_ENV": str(TRAIN_ENV),
    "PATH": str(TRAIN_PYTHON.parent) + os.pathsep + os.environ["PATH"],
    "PYTHONNOUSERSITE": "1",
}
train_env.pop("PYTHONPATH", None)
runtime_lock = PROJECT_DIR / "requirements-runtime.lock"
core_pins = [
    line.split()[0]
    for line in runtime_lock.read_text().splitlines()
    if re.match(r"^[A-Za-z0-9_.-]+==", line)
]
constraints = EVIDENCE_DIR / "runtime-constraints.txt"
constraints.write_text("\n".join(core_pins) + "\n", encoding="utf-8")
install = [
    TRAIN_PYTHON,
    "-u",
    "-m",
    "pip",
    "--isolated",
    "install",
    "--index-url",
    "https://pypi.org/simple",
    "--only-binary=:all:",
]
run_logged(
    "Install hash-locked deployment packages",
    [
        *install,
        "--require-hashes",
        "-r",
        runtime_lock,
    ],
    SESSION_LOG,
    env=train_env,
)
run_logged(
    "Install quality and notebook packages",
    [
        *install,
        "-c",
        constraints,
        "-r",
        "requirements.lock",
        "-r",
        "requirements-ml.lock",
        "-r",
        "requirements-notebook.lock",
    ],
    SESSION_LOG,
    env=train_env,
)
run_logged(
    "Check dependency consistency",
    [
        TRAIN_PYTHON,
        "-u",
        "-m",
        "pip",
        "check",
    ],
    SESSION_LOG,
    env=train_env,
)
quality_passed = False

In [ ]:
verify_runtime = r"""
import hashlib
import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path

if (sys.implementation.name != "cpython" or sys.version_info[:3] != (3, 12, 15)
        or sys.prefix == sys.base_prefix):
    raise RuntimeError("Training requires isolated CPython 3.12.15")
locked = {}
for pin in Path(sys.argv[1]).read_text().splitlines():
    name, expected = pin.split("==")
    locked[name] = version(name)
    if locked[name] != expected:
        raise RuntimeError(f"{name}={locked[name]} differs from deployment pin {expected}")
lock_hash = hashlib.sha256(Path("requirements-runtime.lock").read_bytes()).hexdigest()
print(json.dumps({
    "executable": sys.executable, "python": platform.python_version(),
    "runtime_lock_sha256": lock_hash,
    "runtime_packages": locked,
}, sort_keys=True))
"""
run_logged(
    "Verify and record actual training runtime",
    [
        TRAIN_PYTHON,
        "-u",
        "-c",
        verify_runtime,
        constraints,
    ],
    SESSION_LOG,
    env=train_env,
)

## 3. Nạp đúng dữ liệu phát triển

Chỉ sao chép3file đã phê duyệt và kiểm hash lần nữa. Không tạo validation/test
bằng cách cắt các run cùng ngày ra khỏi bộ TRAIN.


In [ ]:
# Drive and exact source review were checked before installing packages.


In [ ]:
PROCESSED_DIR = PROJECT_DIR / "data/processed-development-v1"
if PROCESSED_DIR.exists():
    raise RuntimeError("Development processed directory already exists; use a fresh runtime")
PROCESSED_DIR.mkdir(parents=True)
for name in EXPECTED_FILES:
    shutil.copy2(DRIVE_DATA_DIR / name, PROCESSED_DIR / name)
for name in EXPECTED_DATA_FILES:
    if hashlib.sha256((PROCESSED_DIR / name).read_bytes()).hexdigest() != expected_hashes[name]:
        raise RuntimeError("Copied development data hash mismatch: " + name)
print("Staged:", EXPECTED_FILES)

## 4. Kế hoạch, nguồn, nhóm thật và EDA

Mỗi family có24run,4benign+4malicious/ngày, ba nhóm hostname/ngày UTC thực.
Pipeline đối chiếu toàn bộ row/plan/source/schema/commitments; không dùng seed,
ID hoặc context mới để gọi các phiên cùng ngày là độc lập. Tất cả dữ liệu là
TRAIN phát triển, không có chỉ số validation/test bên ngoài.


In [ ]:
preflight = r"""
import json
import sys
from collections import Counter
from pathlib import Path
from ml.development_sources import load_training_data
from ml.dataset import load_feature_schema
from ml.training import exploratory_summary

PROJECT_DIR = Path.cwd()
output = Path(sys.argv[2])
rows, manifest = load_training_data(PROJECT_DIR / "data/processed-development-v1",
                                    root=PROJECT_DIR, expected_commit=sys.argv[1])
schema = load_feature_schema(PROJECT_DIR / "ml/feature_schema_http.json")
assert len(rows) == 48 and manifest["record_counts"] == {"train": 48}
assert manifest["external_validation_status"] == manifest["locked_test_status"] == "NOT_COLLECTED"
print("Development strata:", dict(Counter((r["family"], r["label"]) for r in rows)))
eda = exploratory_summary(rows, schema)
eda["independent_host_days"] = {
    family: len({r["evaluation_group_id"] for r in rows if r["family"] == family})
    for family in ("account", "c2")
}
assert set(eda["independent_host_days"].values()) == {3}
(output / "eda.json").write_text(json.dumps(eda), encoding="utf-8")
train_rows = rows
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

for family, feature_order in schema["x-family-features"].items():
    family_rows = [row for row in train_rows if row["family"] == family]
    columns = 3
    rows = (len(feature_order) + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(15, 3.8 * rows))
    axes = np.atleast_1d(axes).ravel()
    for unused in axes[len(feature_order):]:
        unused.set_visible(False)
    for axis, feature in zip(axes[:len(feature_order)], feature_order, strict=True):
        for label, color, name in ((0, "#4C78A8", "benign"), (1, "#E45756", "malicious")):
            values = [row["features"][feature] for row in family_rows if row["label"] == label]
            axis.hist(
                values,
                bins=min(15, max(3, len(set(values)))),
                alpha=0.65,
                label=name,
                color=color,
            )
        axis.set_title(f"{family}: {feature}")
        axis.set_xlabel(feature)
        axis.set_ylabel("count")
        axis.legend()
    plt.tight_layout()
    figure.savefig(output / f"eda-{family}.png")
    plt.close(figure)
"""
run_logged(
    "Verify48 development runs and generate TRAIN EDA",
    [
        TRAIN_PYTHON,
        "-u",
        "-c",
        preflight,
        commit,
        EVIDENCE_DIR,
    ],
    SESSION_LOG,
    env=train_env,
)
manifest = json.loads((PROCESSED_DIR / "dataset_manifest.json").read_bytes())

## 5. EDA phát triển

Đọc phân bố, missing, biến hằng và cân bằng nhãn; số run hiển thị riêng với ba
nhóm máy/ngày. Kết quả phát triển không chứng minh chất lượng ngoài lab.


In [ ]:
import pandas as pd
from IPython.display import Image, display

eda = json.loads((EVIDENCE_DIR / "eda.json").read_bytes())
summary_rows = []
for family, family_stats in eda["families"].items():
    summary_rows.append(
        {
            "family": family,
            "records": family_stats["records"],
            "runs": family_stats["independent_runs"],
            "independent_host_days": eda["independent_host_days"][family],
            "benign": family_stats["benign"],
            "malicious": family_stats["malicious"],
        }
    )
display(pd.DataFrame(summary_rows))

feature_rows = []
for family, family_stats in eda["families"].items():
    for feature, stats in family_stats["features"].items():
        feature_rows.append({"family": family, "feature": feature, **stats})
display(pd.DataFrame(feature_rows))

In [ ]:
for family in eda["families"]:
    display(Image(filename=str(EVIDENCE_DIR / f"eda-{family}.png")))

## 6. Quality gate toàn repo

Ruff và pytest chạy bằng cùng Python cô lập; giữ mọi guard thu/parity/context và
hồi quy. Test phần mềm với dữ liệu giả lập không cấp G7 hay chất lượng model.


In [ ]:
quality_passed = False
if shutil.which("node") is None:
    run_logged("Refresh Node package index", ["apt-get", "update", "-qq"], SESSION_LOG)
    run_logged(
        "Install Node for workflow regression tests",
        [
            "apt-get",
            "install",
            "-y",
            "--no-install-recommends",
            "nodejs",
        ],
        SESSION_LOG,
    )
run_logged("Node version", ["node", "--version"], SESSION_LOG, env=train_env)
run_logged("Ruff", [TRAIN_PYTHON, "-u", "-m", "ruff", "check", "."], SESSION_LOG, env=train_env)
run_logged(
    "Full pytest suite",
    [
        TRAIN_PYTHON,
        "-u",
        "-m",
        "pytest",
        "-o",
        "addopts=",
        "-v",
        "--durations=15",
    ],
    SESSION_LOG,
    env=train_env,
)
quality_passed = True

## 7. Huấn luyện phát triển theo ngày

Outer3fold giữ ra một ngày; chọn model và sigmoid calibration chỉ dùng hai ngày
còn lại, với inner2fold theo ngày. Các fold có thể chọn model khác nhau, vì vậy
OOF là chẩn đoán của quy trình lựa chọn/calibration. Final model được chọn và
refit bằng cả ba ngày; không gán OOF thành điểm test độc lập của final model.

Threshold chọn trên pooled OOF và các F1/precision/FPR tương ứng là chỉ số lựa
chọn phát triển. CI bên ngoài để null; không gọi đây là đánh giá tương lai theo
thời gian. Không cần GPU.


In [ ]:
if not quality_passed:
    raise RuntimeError("The complete quality gate must pass before fitting")
run_logged(
    "Train development models with real day groups",
    [
        TRAIN_PYTHON,
        "-u",
        "-m",
        "ml.train_development",
        "--expected-commit",
        commit,
    ],
    SESSION_LOG,
    env=train_env,
)

## 8. Đọc đúng kết quả phát triển

So với prior baseline và kiểm PR-AUC/F1/FPR/Brier/ECE. Không đổi gate để lấy
điểm đẹp. Không báo48run là48nhóm độc lập. Giá trị uncertainty và CI dưới đây
cố ý phản ánh chưa có đánh giá bên ngoài, kể cả khi điểm CV cao.


In [ ]:
PACKAGE_DIR = PROJECT_DIR / "ml/development-model-v1"
required = (
    "development_model.joblib",
    "metadata.json",
    "metrics_development.json",
    "eda_train.json",
    "feature_schema.json",
)
assert all((PACKAGE_DIR / name).is_file() for name in required)
metadata = json.loads((PACKAGE_DIR / "metadata.json").read_bytes())
metrics = json.loads((PACKAGE_DIR / "metrics_development.json").read_bytes())
assert metadata["artifact_kind"] == "development-model-v1"
assert metadata["dataset_manifest_payload_sha256"] == source_verifier.digest(manifest)
assert metadata["dataset_hash_sha256"] == manifest["dataset_hash_sha256"]
assert metadata["dataset_provenance"] == manifest["provenance"]
assert metadata["external_validation_status"] == metadata["locked_test_status"] == "NOT_COLLECTED"
assert metadata["deployment_eligible"] is metadata["automation_eligible"] is False
summary = []
for family, result in metrics["families"].items():
    m = result["procedure_metrics_at_selected_threshold"]
    assert result["confidence_intervals"] is None
    assert result["uncertainty_status"] == "INSUFFICIENT_EXTERNAL_ASSESSMENT"
    summary.append(
        {
            "family": family,
            "partition": "DEVELOPMENT_OOF_THRESHOLD_SELECTION",
            "final_model": result["selected_model"],
            "host_days": 3,
            **{k: m[k] for k in ("pr_auc", "precision", "recall", "f1", "fpr", "brier", "ece")},
            "CI": None,
            "automation_eligible": False,
        }
    )
    print(
        family,
        "fold-selected models:",
        [f["selected_model"] for f in result["nested_grouped_cv"]["folds"]],
    )
display(pd.DataFrame(summary))
print("Model:", metadata["model_version"])
print("External validation/test:", metadata["external_validation_status"])

## 9. Lưu artifact phát triển vào Drive

Archive chứa mô hình offline, metadata, chẩn đoán CV, EDA và log. Không chứa raw,
khóa hoặc tập test. File xuất riêng theo model_version, không ghi đè phiên trước.


In [ ]:
import tarfile
from datetime import UTC, datetime

experiment_summary = {
    "experiment": "development-three-day-v1",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "git_commit": commit,
    "model_version": metadata["model_version"],
    "dataset_hash_sha256": metadata["dataset_hash_sha256"],
    "dataset_manifest_payload_sha256": metadata["dataset_manifest_payload_sha256"],
    "external_validation_status": "NOT_COLLECTED",
    "locked_test_status": "NOT_COLLECTED",
    "deployment_eligible": False,
    "automation_eligible": False,
}
summary_path = EVIDENCE_DIR / "development-experiment-summary.json"
summary_path.write_text(json.dumps(experiment_summary, indent=2) + "\n", encoding="utf-8")
archive_path = PROJECT_DIR.parent / f"development-model-{metadata['model_version']}.tar.gz"
with tarfile.open(archive_path, "x:gz") as archive:
    archive.add(PACKAGE_DIR, arcname="development-model")
    archive.add(SESSION_LOG, arcname="evidence/development-training.log")
    archive.add(summary_path, arcname="evidence/development-experiment-summary.json")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
destination = DRIVE_OUTPUT_DIR / archive_path.name
with archive_path.open("rb") as incoming, destination.open("xb") as target:
    shutil.copyfileobj(incoming, target)
archive_hash = hashlib.sha256(archive_path.read_bytes()).hexdigest()
if hashlib.sha256(destination.read_bytes()).hexdigest() != archive_hash:
    raise RuntimeError("Drive archive checksum mismatch")
print("Saved development artifact:", destination)
print("Archive SHA256:", archive_hash)
print(json.dumps(experiment_summary, indent=2))

## 10. Hoàn tất thí nghiệm phát triển

Hoàn tất khi quality gate, kiểm dữ liệu và fit đều thành công, archive có trên
Drive và SHA khớp. Giữ source SHA, manifest SHA, model_version và log làm bằng
chứng. Đây là artifact offline phát triển: production ModelRuntime cố ý từ chối,
không gọi freeze_model hoặc đổi tên file để giả thành package đã nghiệm thu.

Chỉ load joblib do phiên train tin cậy của chính nhóm tạo. Có thể dùng nó cho
phân tích/scoring offline; không bật phản ứng tự động. Kết quả thực tế có thể
yếu dù train hoàn tất, phải báo nguyên trạng. Chưa có validation/test bên ngoài
hoặc chứng cứ tổng quát hóa; không dùng CV nhỏ để tuyên bố đã giải quyết điều đó.
